# Notebook 4: Deep Learning Models (LSTM & Bidirectional LSTM)

### Objective:
1. Tokenize texts into sequences with PyTorch Embedding layers.
2. Implement LSTM and Bidirectional LSTM with temporal max-pooling.
3. Handle class imbalance using weighted BCE Loss.
4. Evaluate deep learning performance against baseline benchmarks.

In [ ]:
import os, sys, json, torch
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

sys.path.append('..')
from src.train_models import LSTMClassifier, BiLSTMClassifier, TextDataset

## 1. Load PyTorch Models

In [ ]:
with open('../models/vocab.json', 'r') as f:
    vocab = json.load(f)
print('Vocabulary size:', len(vocab))

test_df = pd.read_csv('../data/processed/test.csv')
y_test = (test_df['label'] == 'spam').astype(int).values

test_ds = TextDataset(test_df['message'], y_test, vocab, max_len=100)

lstm = LSTMClassifier(len(vocab))
lstm.load_state_dict(torch.load('../models/lstm_model.pt', map_location='cpu'))
lstm.eval()

bilstm = BiLSTMClassifier(len(vocab))
bilstm.load_state_dict(torch.load('../models/bilstm_model.pt', map_location='cpu'))
bilstm.eval()
print('Models loaded successfully!')

## 2. Evaluate Bidirectional LSTM

In [ ]:
with torch.no_grad():
    logits = bilstm(test_ds.data).squeeze()
    probs = torch.sigmoid(logits).numpy()
preds = (probs >= 0.5).astype(int)

print('--- Bidirectional LSTM Evaluation ---')
print(classification_report(y_test, preds, target_names=['Ham', 'Spam']))